In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('../data/studytrack_clustered.csv')
df.shape

(80000, 34)

In [10]:
cat_cols = ['gender', 'major', 'part_time_job', 'diet_quality', 'parental_education_level',
            'internet_quality', 'extracurricular_participation', 'dropout_risk',
            'study_environment', 'access_to_tutoring', 'family_income_range', 'learning_style']

df_model = df.copy()
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    encoders[col] = le

print("Encoded columns:", cat_cols)

Encoded columns: ['gender', 'major', 'part_time_job', 'diet_quality', 'parental_education_level', 'internet_quality', 'extracurricular_participation', 'dropout_risk', 'study_environment', 'access_to_tutoring', 'family_income_range', 'learning_style']


In [11]:
feature_cols = ['age', 'study_hours_per_day', 'social_media_hours', 'netflix_hours',
                 'part_time_job', 'attendance_percentage', 'sleep_hours', 'diet_quality',
                 'exercise_frequency', 'parental_education_level', 'internet_quality',
                 'mental_health_rating', 'extracurricular_participation', 'previous_gpa',
                 'stress_level', 'social_activity', 'screen_time', 'study_environment',
                 'access_to_tutoring', 'family_income_range', 'parental_support_level',
                 'motivation_level', 'exam_anxiety_score', 'learning_style',
                 'time_management_score', 'cluster']

X = df_model[feature_cols]
y = df_model['exam_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

Train shape: (64000, 26) Test shape: (16000, 26)


In [12]:
habit_feature_cols = [c for c in feature_cols if c != 'previous_gpa']

X_habit = df_model[habit_feature_cols]
y_habit = df_model['exam_score']

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_habit, y_habit, test_size=0.2, random_state=42)

In [13]:
import joblib
import json

with open('../models/feature_columns.json', 'w') as f:
    json.dump({
        'full_model_features': feature_cols,
        'habit_model_features': habit_feature_cols
    }, f, indent=2)

# Save encoders so the backend can transform new user input the same way
joblib.dump(encoders, '../models/label_encoders.pkl')

print("Feature columns and encoders saved.")

Feature columns and encoders saved.


In [14]:
print(df_model['dropout_risk'].value_counts())
print(df_model['dropout_risk'].value_counts(normalize=True))

dropout_risk
0    78418
1     1582
Name: count, dtype: int64
dropout_risk
0    0.980225
1    0.019775
Name: proportion, dtype: float64


In [15]:
print(encoders['dropout_risk'].classes_)

['No' 'Yes']


In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Exclude previous_gpa and exam_score to keep this focused on behavioral/lifestyle risk signals
# (a student's risk should be flagged from habits, not just restating their grades)
risk_feature_cols = [c for c in habit_feature_cols if c not in ['cluster']]

X_risk = df_model[risk_feature_cols]
y_risk = df_model['dropout_risk']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_risk, y_risk, test_size=0.2, random_state=42, stratify=y_risk
)

print("Train shape:", X_train_r.shape, "Test shape:", X_test_r.shape)
print("Train class balance:\n", y_train_r.value_counts(normalize=True))

Train shape: (64000, 24) Test shape: (16000, 24)
Train class balance:
 dropout_risk
0    0.980219
1    0.019781
Name: proportion, dtype: float64


In [17]:
rf_classifier = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_classifier.fit(X_train_r, y_train_r)
print("Classifier trained.")

Classifier trained.


In [18]:
y_pred_r = rf_classifier.predict(X_test_r)

print(classification_report(y_test_r, y_pred_r, target_names=['No Risk', 'At Risk']))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_r, y_pred_r))

              precision    recall  f1-score   support

     No Risk       1.00      1.00      1.00     15684
     At Risk       1.00      1.00      1.00       316

    accuracy                           1.00     16000
   macro avg       1.00      1.00      1.00     16000
weighted avg       1.00      1.00      1.00     16000


Confusion Matrix:
[[15684     0]
 [    0   316]]


In [19]:
risk_importances = pd.DataFrame({
    'feature': risk_feature_cols,
    'importance': rf_classifier.feature_importances_
}).sort_values('importance', ascending=False)

print(risk_importances.to_string(index=False))

                      feature  importance
                 stress_level    0.596612
             motivation_level    0.245385
           exam_anxiety_score    0.116423
         mental_health_rating    0.012977
        attendance_percentage    0.003298
        time_management_score    0.002823
                  screen_time    0.002757
          study_hours_per_day    0.002734
                  sleep_hours    0.002555
           social_media_hours    0.002418
                netflix_hours    0.002142
                          age    0.001987
       parental_support_level    0.001227
           exercise_frequency    0.001045
              social_activity    0.000950
            study_environment    0.000907
     parental_education_level    0.000890
               learning_style    0.000597
                 diet_quality    0.000569
             internet_quality    0.000515
          family_income_range    0.000404
extracurricular_participation    0.000298
                part_time_job    0

In [20]:
for col in ['attendance_percentage', 'mental_health_rating', 'stress_level', 'exam_anxiety_score', 'time_management_score']:
    print(f"\n--- {col} vs dropout_risk ---")
    print(df.groupby('dropout_risk')[col].describe()[['mean', 'min', 'max']])


--- attendance_percentage vs dropout_risk ---
                   mean   min    max
dropout_risk                        
No            69.983339  40.0  100.0
Yes           69.201770  40.0   99.9

--- mental_health_rating vs dropout_risk ---
                  mean  min   max
dropout_risk                     
No            6.819957  1.0  10.0
Yes           6.018458  1.0  10.0

--- stress_level vs dropout_risk ---
                  mean  min   max
dropout_risk                     
No            4.934265  1.0  10.0
Yes           8.889381  8.1  10.0

--- exam_anxiety_score vs dropout_risk ---
                   mean   min   max
dropout_risk                       
No             8.478385   5.0  10.0
Yes           10.000000  10.0  10.0

--- time_management_score vs dropout_risk ---
                  mean  min   max
dropout_risk                     
No            5.500048  1.0  10.0
Yes           5.453729  1.0  10.0


In [21]:
def calculate_risk_factors(row):
    factors = []
    if row['stress_level'] >= 8:
        factors.append('High stress level')
    if row['exam_anxiety_score'] >= 8:
        factors.append('High exam anxiety')
    if row['motivation_level'] <= 4:
        factors.append('Low motivation')
    if row['attendance_percentage'] < 75:
        factors.append('Low attendance')
    if row['mental_health_rating'] <= 4:
        factors.append('Low mental health rating')
    if row['sleep_hours'] < 6:
        factors.append('Insufficient sleep')
    return factors

df['risk_factors'] = df.apply(calculate_risk_factors, axis=1)
df['risk_factor_count'] = df['risk_factors'].apply(len)

df['risk_factor_count'].value_counts().sort_index()

risk_factor_count
0     6411
1    18738
2    25610
3    21421
4     6900
5      874
6       46
Name: count, dtype: int64

In [22]:
def assign_risk_level(count):
    if count <= 1:
        return 'Low'
    elif count <= 3:
        return 'Moderate'
    else:
        return 'High'

df['risk_level'] = df['risk_factor_count'].apply(assign_risk_level)
print(df['risk_level'].value_counts())
print("\nAvg exam_score by risk_level:")
print(df.groupby('risk_level')['exam_score'].mean().sort_values(ascending=False))

risk_level
Moderate    47031
Low         25149
High         7820
Name: count, dtype: int64

Avg exam_score by risk_level:
risk_level
Low         92.304664
Moderate    88.165083
High        84.839642
Name: exam_score, dtype: float64


In [23]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import joblib

# Retrain with a lighter configuration — still accurate, much smaller file size
rf_model_light = RandomForestRegressor(
    n_estimators=100, max_depth=10, min_samples_split=10, random_state=42, n_jobs=-1
)
rf_model_light.fit(X_train, y_train)

rf_habit_model_light = RandomForestRegressor(
    n_estimators=100, max_depth=10, min_samples_split=10, random_state=42, n_jobs=-1
)
rf_habit_model_light.fit(X_train_h, y_train_h)

# Re-check accuracy hasn't dropped meaningfully
print("Full model  -> MAE:", mean_absolute_error(y_test, rf_model_light.predict(X_test)),
      "R2:", r2_score(y_test, rf_model_light.predict(X_test)))
print("Habit model -> MAE:", mean_absolute_error(y_test_h, rf_habit_model_light.predict(X_test_h)),
      "R2:", r2_score(y_test_h, rf_habit_model_light.predict(X_test_h)))

# Save WITH compression this time — dramatically reduces file size
joblib.dump(rf_model_light, '../models/exam_score_predictor_full.pkl', compress=3)
joblib.dump(rf_habit_model_light, '../models/exam_score_predictor_habits.pkl', compress=3)

print("Lightweight compressed models saved.")

Full model  -> MAE: 3.2291054666568524 R2: 0.8714080484086448
Habit model -> MAE: 8.608116706906385 R2: 0.18657740605796735
Lightweight compressed models saved.
